# AI Tournament Analysis

This notebook provides comprehensive analysis of the round-robin tournament results where each AI player type competes as a team (both players on a team are the same type).


In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Dict, List, Optional, Tuple

# Set up project path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries loaded successfully")


## Setup & Data Loading

Load tournament results from JSON file. The script searches for the most recent tournament results file.


In [ ]:
# Search for tournament result files
tournament_files = list(Path(".").glob("tournament_results_*.json"))
if not tournament_files:
    # Also check project root
    tournament_files = list(project_root.glob("tournament_results_*.json"))

if tournament_files:
    # Get the most recent file
    latest_file = max(tournament_files, key=lambda p: p.stat().st_mtime)
    with open(latest_file, 'r') as f:
        tournament_data = json.load(f)
    print(f"Loaded tournament results: {latest_file.name}")
    print(f"Timestamp: {tournament_data.get('timestamp', 'Unknown')}")
    print(f"Player types tested: {', '.join(tournament_data.get('player_types_tested', []))}")
    print(f"Games per matchup: {tournament_data.get('num_games_per_matchup', 'Unknown')}")
    print(f"Total games: {tournament_data.get('total_games', 'Unknown')}")
else:
    print("No tournament result files found. Please run scripts/ai_tournament.py first.")
    tournament_data = None


## Tournament Overview

Summary statistics and overview of the tournament structure.


In [ ]:
if tournament_data:
    print("=" * 80)
    print("TOURNAMENT OVERVIEW")
    print("=" * 80)
    print()
    
    print(f"Total Games Played: {tournament_data.get('total_games', 0):,}")
    print(f"Total Matchups: {tournament_data.get('total_matchups', 0)}")
    print(f"Games per Matchup: {tournament_data.get('num_games_per_matchup', 0)}")
    print(f"Player Types Tested: {len(tournament_data.get('player_types_tested', []))}")
    print()
    
    # Calculate expected matchups (round-robin: n*(n-1)/2)
    n_types = len(tournament_data.get('player_types_tested', []))
    expected_matchups = n_types * (n_types - 1) // 2
    print(f"Expected Matchups (round-robin): {expected_matchups}")
    print(f"Actual Matchups: {tournament_data.get('total_matchups', 0)}")
    print()
    
    # Show player types
    print("Player Types in Tournament:")
    for i, pt in enumerate(tournament_data.get('player_types_tested', []), 1):
        print(f"  {i}. {pt}")
else:
    print("No tournament data available")


## Player Type Rankings

Rankings of player types by win rate, with detailed statistics.


In [ ]:
if tournament_data:
    rankings = tournament_data.get('player_type_rankings', [])
    
    if rankings:
        # Create DataFrame for easier analysis
        df_rankings = pd.DataFrame(rankings)
        
        print("=" * 80)
        print("PLAYER TYPE RANKINGS")
        print("=" * 80)
        print()
        
        # Display rankings table
        display_cols = ['player_type', 'win_rate', 'wins', 'losses', 'ties', 
                        'total_games', 'avg_score', 'avg_score_against', 'avg_score_diff']
        df_display = df_rankings[display_cols].copy()
        df_display['win_rate'] = df_display['win_rate'].apply(lambda x: f"{x:.1%}")
        df_display['avg_score'] = df_display['avg_score'].apply(lambda x: f"{x:.2f}")
        df_display['avg_score_against'] = df_display['avg_score_against'].apply(lambda x: f"{x:.2f}")
        df_display['avg_score_diff'] = df_display['avg_score_diff'].apply(lambda x: f"{x:+.2f}")
        
        print(df_display.to_string(index=False))
        
        # Store for later use
        rankings_df = df_rankings
    else:
        print("No ranking data available")
        rankings_df = None
else:
    rankings_df = None


## Head-to-Head Analysis

Matrix showing win rates between each pair of player types.


In [ ]:
if tournament_data:
    matchups = tournament_data.get('matchups', {})
    
    if matchups:
        # Extract player types
        player_types = tournament_data.get('player_types_tested', [])
        
        # Create head-to-head matrix
        h2h_data = []
        for matchup_key, matchup_stats in matchups.items():
            # Parse matchup key: "type0_vs_type1"
            parts = matchup_key.split('_vs_')
            if len(parts) == 2:
                type0, type1 = parts[0], parts[1]
                
                games = matchup_stats.get('games_played', 0)
                if games > 0:
                    # Determine which type is which based on team wins
                    # We need to check the original matchup to know which type was team0
                    team0_wins = matchup_stats.get('team0_wins', 0)
                    team1_wins = matchup_stats.get('team1_wins', 0)
                    
                    # For each matchup, record both directions
                    # We'll need to check the original data structure
                    # For now, assume type0 was team0 and type1 was team1
                    h2h_data.append({
                        'player_type_0': type0,
                        'player_type_1': type1,
                        'type0_wins': team0_wins,
                        'type1_wins': team1_wins,
                        'games': games,
                        'type0_win_rate': team0_wins / games if games > 0 else 0.0,
                        'type1_win_rate': team1_wins / games if games > 0 else 0.0,
                    })
        
        if h2h_data:
            df_h2h = pd.DataFrame(h2h_data)
            
            # Create win rate matrix
            win_rate_matrix = pd.DataFrame(
                index=player_types,
                columns=player_types,
                dtype=float
            )
            
            # Fill matrix (symmetric)
            for _, row in df_h2h.iterrows():
                type0 = row['player_type_0']
                type1 = row['player_type_1']
                type0_wr = row['type0_win_rate']
                
                win_rate_matrix.loc[type0, type1] = type0_wr
                win_rate_matrix.loc[type1, type0] = 1.0 - type0_wr
            
            # Fill diagonal with 0.5 (self-matchup, though shouldn't happen in round-robin)
            for pt in player_types:
                win_rate_matrix.loc[pt, pt] = 0.5
            
            print("Head-to-Head Win Rate Matrix")
            print("(Row player type win rate vs column player type)")
            print()
            print(win_rate_matrix.round(3).to_string())
            
            # Store for visualization
            h2h_matrix = win_rate_matrix
        else:
            h2h_matrix = None
    else:
        h2h_matrix = None
else:
    h2h_matrix = None


## Statistical Significance

Calculate confidence intervals and perform statistical tests to determine if win rate differences are significant.


In [ ]:
if rankings_df is not None and len(rankings_df) > 0:
    print("=" * 80)
    print("STATISTICAL SIGNIFICANCE ANALYSIS")
    print("=" * 80)
    print()
    
    # Calculate confidence intervals for win rates (95% CI using normal approximation)
    def calculate_ci(wins: int, total: int, confidence: float = 0.95) -> Tuple[float, float]:
        """Calculate confidence interval for win rate."""
        if total == 0:
            return (0.0, 0.0)
        p = wins / total
        z = stats.norm.ppf((1 + confidence) / 2)
        margin = z * np.sqrt(p * (1 - p) / total)
        return (max(0, p - margin), min(1, p + margin))
    
    # Add confidence intervals to rankings
    rankings_df['ci_lower'] = rankings_df.apply(
        lambda row: calculate_ci(row['wins'], row['total_games'])[0], axis=1
    )
    rankings_df['ci_upper'] = rankings_df.apply(
        lambda row: calculate_ci(row['wins'], row['total_games'])[1], axis=1
    )
    rankings_df['ci_width'] = rankings_df['ci_upper'] - rankings_df['ci_lower']
    
    print("Win Rate Confidence Intervals (95%):")
    print()
    for _, row in rankings_df.iterrows():
        print(f"{row['player_type']}:")
        print(f"  Win Rate: {row['win_rate']:.1%} (95% CI: {row['ci_lower']:.1%} - {row['ci_upper']:.1%})")
        print(f"  CI Width: {row['ci_width']:.1%}")
        print()
    
    # Perform pairwise comparisons (if we have matchup data)
    if tournament_data and 'matchups' in tournament_data:
        print("=" * 80)
        print("PAIRWISE STATISTICAL TESTS")
        print("=" * 80)
        print()
        
        # Binomial tests for each matchup
        matchups = tournament_data['matchups']
        significant_matchups = []
        
        for matchup_key, matchup_stats in matchups.items():
            games = matchup_stats.get('games_played', 0)
            if games >= 10:  # Only test if enough games
                team0_wins = matchup_stats.get('team0_wins', 0)
                team1_wins = matchup_stats.get('team1_wins', 0)
                
                # Binomial test: is win rate significantly different from 0.5?
                p_value = stats.binom_test(team0_wins, games, p=0.5, alternative='two-sided')
                
                if p_value < 0.05:
                    significant_matchups.append({
                        'matchup': matchup_key,
                        'team0_wins': team0_wins,
                        'team1_wins': team1_wins,
                        'p_value': p_value,
                    })
        
        if significant_matchups:
            print(f"Found {len(significant_matchups)} matchups with statistically significant results (p < 0.05):")
            print()
            for sig in significant_matchups[:10]:  # Show top 10
                print(f"{sig['matchup']}:")
                print(f"  Team 0: {sig['team0_wins']} wins, Team 1: {sig['team1_wins']} wins")
                print(f"  p-value: {sig['p_value']:.4f}")
                print()
        else:
            print("No matchups with statistically significant results (p < 0.05)")
else:
    print("No ranking data available for statistical analysis")


## Visualizations

Comprehensive visualizations of tournament results.


In [ ]:
if rankings_df is not None and len(rankings_df) > 0:
    # Create figure with subplots
    fig = plt.figure(figsize=(20, 16))
    
    # 1. Win Rate Bar Chart (Ranked)
    ax1 = plt.subplot(3, 3, 1)
    df_sorted = rankings_df.sort_values('win_rate', ascending=True)
    colors = plt.cm.viridis(np.linspace(0, 1, len(df_sorted)))
    bars = ax1.barh(range(len(df_sorted)), df_sorted['win_rate'], color=colors)
    ax1.set_yticks(range(len(df_sorted)))
    ax1.set_yticklabels(df_sorted['player_type'])
    ax1.set_xlabel('Win Rate', fontsize=12)
    ax1.set_title('Player Type Rankings by Win Rate', fontsize=14, fontweight='bold')
    ax1.set_xlim(0, 1)
    ax1.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, (idx, row) in enumerate(df_sorted.iterrows()):
        ax1.text(row['win_rate'] + 0.01, i, f"{row['win_rate']:.1%}", 
                va='center', fontsize=10)
    
    # 2. Head-to-Head Heatmap
    if h2h_matrix is not None:
        ax2 = plt.subplot(3, 3, 2)
        sns.heatmap(h2h_matrix, annot=True, fmt='.2f', cmap='RdYlGn', 
                   center=0.5, vmin=0, vmax=1, ax=ax2, 
                   cbar_kws={'label': 'Win Rate'}, square=True)
        ax2.set_xlabel('Opponent Player Type', fontsize=12)
        ax2.set_ylabel('Player Type', fontsize=12)
        ax2.set_title('Head-to-Head Win Rate Matrix', fontsize=14, fontweight='bold')
        plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
        plt.setp(ax2.get_yticklabels(), rotation=0)
    
    # 3. Score Distribution
    ax3 = plt.subplot(3, 3, 3)
    if tournament_data and 'matchups' in tournament_data:
        all_scores_team0 = []
        all_scores_team1 = []
        for matchup_stats in tournament_data['matchups'].values():
            all_scores_team0.extend(matchup_stats.get('scores_team0', []))
            all_scores_team1.extend(matchup_stats.get('scores_team1', []))
        
        if all_scores_team0 and all_scores_team1:
            ax3.hist(all_scores_team0, bins=20, alpha=0.6, label='Team 0', density=True)
            ax3.hist(all_scores_team1, bins=20, alpha=0.6, label='Team 1', density=True)
            ax3.set_xlabel('Final Score', fontsize=12)
            ax3.set_ylabel('Density', fontsize=12)
            ax3.set_title('Score Distribution', fontsize=14, fontweight='bold')
            ax3.legend()
            ax3.grid(alpha=0.3)
    
    # 4. Average Score vs Win Rate
    ax4 = plt.subplot(3, 3, 4)
    ax4.scatter(rankings_df['avg_score'], rankings_df['win_rate'], 
               s=100, alpha=0.6, c=range(len(rankings_df)), cmap='viridis')
    ax4.set_xlabel('Average Score', fontsize=12)
    ax4.set_ylabel('Win Rate', fontsize=12)
    ax4.set_title('Average Score vs Win Rate', fontsize=14, fontweight='bold')
    ax4.grid(alpha=0.3)
    
    # Add labels
    for _, row in rankings_df.iterrows():
        ax4.annotate(row['player_type'], 
                    (row['avg_score'], row['win_rate']),
                    fontsize=8, alpha=0.7)
    
    # 5. Score Difference Distribution
    ax5 = plt.subplot(3, 3, 5)
    if 'avg_score_diff' in rankings_df.columns:
        ax5.barh(range(len(rankings_df)), rankings_df.sort_values('avg_score_diff')['avg_score_diff'],
                color=plt.cm.RdYlGn(np.linspace(0, 1, len(rankings_df))))
        df_sorted_diff = rankings_df.sort_values('avg_score_diff', ascending=True)
        ax5.set_yticks(range(len(df_sorted_diff)))
        ax5.set_yticklabels(df_sorted_diff['player_type'])
        ax5.set_xlabel('Average Score Difference', fontsize=12)
        ax5.set_title('Score Difference (For - Against)', fontsize=14, fontweight='bold')
        ax5.axvline(x=0, color='red', linestyle='--', alpha=0.5)
        ax5.grid(axis='x', alpha=0.3)
    
    # 6. Hands per Game Analysis
    ax6 = plt.subplot(3, 3, 6)
    if tournament_data and 'matchups' in tournament_data:
        all_hands = []
        for matchup_stats in tournament_data['matchups'].values():
            all_hands.extend(matchup_stats.get('hands_played', []))
        
        if all_hands:
            ax6.hist(all_hands, bins=30, alpha=0.7, edgecolor='black')
            ax6.set_xlabel('Hands per Game', fontsize=12)
            ax6.set_ylabel('Frequency', fontsize=12)
            ax6.set_title('Distribution of Hands per Game', fontsize=14, fontweight='bold')
            ax6.axvline(np.mean(all_hands), color='red', linestyle='--', 
                      label=f'Mean: {np.mean(all_hands):.1f}')
            ax6.legend()
            ax6.grid(alpha=0.3)
    
    # 7. Win Rate Confidence Intervals
    ax7 = plt.subplot(3, 3, 7)
    if 'ci_lower' in rankings_df.columns:
        df_sorted_ci = rankings_df.sort_values('win_rate', ascending=True)
        y_pos = range(len(df_sorted_ci))
        ax7.errorbar(df_sorted_ci['win_rate'], y_pos, 
                    xerr=[df_sorted_ci['win_rate'] - df_sorted_ci['ci_lower'],
                          df_sorted_ci['ci_upper'] - df_sorted_ci['win_rate']],
                    fmt='o', capsize=5, capthick=2)
        ax7.set_yticks(y_pos)
        ax7.set_yticklabels(df_sorted_ci['player_type'])
        ax7.set_xlabel('Win Rate', fontsize=12)
        ax7.set_title('Win Rate with 95% Confidence Intervals', fontsize=14, fontweight='bold')
        ax7.set_xlim(0, 1)
        ax7.grid(axis='x', alpha=0.3)
    
    # 8. Wins vs Losses
    ax8 = plt.subplot(3, 3, 8)
    ax8.scatter(rankings_df['wins'], rankings_df['losses'], 
               s=100, alpha=0.6, c=rankings_df['win_rate'], cmap='viridis')
    ax8.set_xlabel('Wins', fontsize=12)
    ax8.set_ylabel('Losses', fontsize=12)
    ax8.set_title('Wins vs Losses', fontsize=14, fontweight='bold')
    ax8.grid(alpha=0.3)
    cbar = plt.colorbar(ax8.collections[0], ax=ax8)
    cbar.set_label('Win Rate', fontsize=10)
    
    # Add labels
    for _, row in rankings_df.iterrows():
        ax8.annotate(row['player_type'], 
                    (row['wins'], row['losses']),
                    fontsize=8, alpha=0.7)
    
    # 9. Total Games Played
    ax9 = plt.subplot(3, 3, 9)
    df_sorted_games = rankings_df.sort_values('total_games', ascending=True)
    ax9.barh(range(len(df_sorted_games)), df_sorted_games['total_games'], 
            color=plt.cm.plasma(np.linspace(0, 1, len(df_sorted_games))))
    ax9.set_yticks(range(len(df_sorted_games)))
    ax9.set_yticklabels(df_sorted_games['player_type'])
    ax9.set_xlabel('Total Games Played', fontsize=12)
    ax9.set_title('Games Played per Player Type', fontsize=14, fontweight='bold')
    ax9.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Visualizations complete!")
else:
    print("No data available for visualization")


## Detailed Matchup Analysis

Deep dive into specific matchups with detailed statistics.


In [ ]:
if tournament_data and 'matchups' in tournament_data:
    matchups = tournament_data['matchups']
    
    print("=" * 80)
    print("DETAILED MATCHUP ANALYSIS")
    print("=" * 80)
    print()
    
    # Create detailed matchup DataFrame
    matchup_details = []
    for matchup_key, matchup_stats in matchups.items():
        games = matchup_stats.get('games_played', 0)
        if games > 0:
            team0_wins = matchup_stats.get('team0_wins', 0)
            team1_wins = matchup_stats.get('team1_wins', 0)
            team0_wr = team0_wins / games
            team1_wr = team1_wins / games
            
            hands_played = matchup_stats.get('hands_played', [])
            avg_hands = np.mean(hands_played) if hands_played else 0.0
            std_hands = np.std(hands_played) if hands_played else 0.0
            
            scores_team0 = matchup_stats.get('scores_team0', [])
            scores_team1 = matchup_stats.get('scores_team1', [])
            avg_score_team0 = np.mean(scores_team0) if scores_team0 else 0.0
            avg_score_team1 = np.mean(scores_team1) if scores_team1 else 0.0
            
            matchup_details.append({
                'matchup': matchup_key,
                'games': games,
                'team0_wins': team0_wins,
                'team1_wins': team1_wins,
                'team0_win_rate': team0_wr,
                'team1_win_rate': team1_wr,
                'avg_hands': avg_hands,
                'std_hands': std_hands,
                'avg_score_team0': avg_score_team0,
                'avg_score_team1': avg_score_team1,
            })
    
    if matchup_details:
        df_matchups = pd.DataFrame(matchup_details)
        
        # Sort by win rate difference (most competitive to least)
        df_matchups['win_rate_diff'] = abs(df_matchups['team0_win_rate'] - df_matchups['team1_win_rate'])
        
        print("All Matchups (sorted by competitiveness):")
        print()
        df_display = df_matchups.sort_values('win_rate_diff')[['matchup', 'games', 'team0_wins', 
                                                               'team1_wins', 'team0_win_rate', 
                                                               'team1_win_rate', 'avg_hands']].copy()
        df_display['team0_win_rate'] = df_display['team0_win_rate'].apply(lambda x: f"{x:.1%}")
        df_display['team1_win_rate'] = df_display['team1_win_rate'].apply(lambda x: f"{x:.1%}")
        df_display['avg_hands'] = df_display['avg_hands'].apply(lambda x: f"{x:.1f}")
        print(df_display.to_string(index=False))
        
        # Most competitive matchups (closest to 50/50)
        print()
        print("=" * 80)
        print("MOST COMPETITIVE MATCHUPS (closest to 50/50)")
        print("=" * 80)
        print()
        top_competitive = df_matchups.nsmallest(5, 'win_rate_diff')
        for _, row in top_competitive.iterrows():
            print(f"{row['matchup']}:")
            print(f"  Team 0 Win Rate: {row['team0_win_rate']:.1%}")
            print(f"  Team 1 Win Rate: {row['team1_win_rate']:.1%}")
            print(f"  Games: {row['games']}")
            print(f"  Avg Hands per Game: {row['avg_hands']:.1f}")
            print()
        
        # Most lopsided matchups
        print("=" * 80)
        print("MOST LOP-SIDED MATCHUPS")
        print("=" * 80)
        print()
        top_lopsided = df_matchups.nlargest(5, 'win_rate_diff')
        for _, row in top_lopsided.iterrows():
            print(f"{row['matchup']}:")
            print(f"  Team 0 Win Rate: {row['team0_win_rate']:.1%}")
            print(f"  Team 1 Win Rate: {row['team1_win_rate']:.1%}")
            print(f"  Games: {row['games']}")
            print(f"  Avg Hands per Game: {row['avg_hands']:.1f}")
            print()
else:
    print("No matchup data available")


## Conclusions

Summary of key findings from the tournament analysis.


In [ ]:
if rankings_df is not None and len(rankings_df) > 0:
    print("=" * 80)
    print("TOURNAMENT CONCLUSIONS")
    print("=" * 80)
    print()
    
    # Top performer
    top_performer = rankings_df.iloc[0]
    print(f"🏆 Top Performer: {top_performer['player_type']}")
    print(f"   Win Rate: {top_performer['win_rate']:.1%}")
    print(f"   Record: {int(top_performer['wins'])}-{int(top_performer['losses'])}-{int(top_performer['ties'])}")
    print(f"   Average Score: {top_performer['avg_score']:.2f}")
    print()
    
    # Bottom performer
    bottom_performer = rankings_df.iloc[-1]
    print(f"📉 Lowest Performer: {bottom_performer['player_type']}")
    print(f"   Win Rate: {bottom_performer['win_rate']:.1%}")
    print(f"   Record: {int(bottom_performer['wins'])}-{int(bottom_performer['losses'])}-{int(bottom_performer['ties'])}")
    print(f"   Average Score: {bottom_performer['avg_score']:.2f}")
    print()
    
    # Win rate spread
    win_rate_spread = top_performer['win_rate'] - bottom_performer['win_rate']
    print(f"📊 Win Rate Spread: {win_rate_spread:.1%}")
    print()
    
    # Most consistent (smallest CI width)
    if 'ci_width' in rankings_df.columns:
        most_consistent = rankings_df.nsmallest(1, 'ci_width').iloc[0]
        print(f"🎯 Most Consistent (narrowest CI): {most_consistent['player_type']}")
        print(f"   CI Width: {most_consistent['ci_width']:.1%}")
        print()
    
    # Best score differential
    if 'avg_score_diff' in rankings_df.columns:
        best_diff = rankings_df.nlargest(1, 'avg_score_diff').iloc[0]
        print(f"⚡ Best Score Differential: {best_diff['player_type']}")
        print(f"   Average Score Difference: {best_diff['avg_score_diff']:+.2f}")
        print()
    
    # Tournament statistics
    if tournament_data:
        print("=" * 80)
        print("TOURNAMENT STATISTICS")
        print("=" * 80)
        print()
        print(f"Total Games: {tournament_data.get('total_games', 0):,}")
        print(f"Total Matchups: {tournament_data.get('total_matchups', 0)}")
        print(f"Games per Matchup: {tournament_data.get('num_games_per_matchup', 0)}")
        
        # Average hands per game
        if 'matchups' in tournament_data:
            all_hands = []
            for matchup_stats in tournament_data['matchups'].values():
                all_hands.extend(matchup_stats.get('hands_played', []))
            if all_hands:
                print(f"Average Hands per Game: {np.mean(all_hands):.1f}")
                print(f"Std Dev Hands per Game: {np.std(all_hands):.1f}")
    
    print()
    print("=" * 80)
    print("Analysis Complete!")
    print("=" * 80)
else:
    print("No data available for conclusions")

